# 당뇨병 데이터 분석 — 연습용(practice)

scikit-learn 당뇨병 데이터셋을 분석하는 실습.
심화 EDA → 피처 엔지니어링 → 데이터 증강(엄격 비교) → 다중 모델·튜닝 → 모델 해석의 전체 파이프라인 구성.

**실습 방법**: `# TODO` 빈칸(`______`)을 채운 뒤 셀 실행. 막히면 답지용과 비교.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import skew, kurtosis
from IPython.display import display

# 한글 폰트 자동 선택 (Mac/Linux/Win 호환)
for cand in ["AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Malgun Gothic", "NanumBarunGothic"]:
    if any(cand in f.name for f in fm.fontManager.ttflist):
        plt.rcParams["font.family"] = cand
        break
plt.rcParams["axes.unicode_minus"] = False

from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, IsolationForest)
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (train_test_split, KFold, RepeatedKFold,
                                     cross_val_score, RandomizedSearchCV, learning_curve)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.mixture import GaussianMixture

RANDOM_STATE = 42
print("준비 완료")

## 1. 데이터 로드 및 품질 점검

결측·중복 점검과 요약 통계로 데이터 상태 파악.

In [ ]:
# [TODO 1] 원본 스케일로 데이터 로드 후 sex 를 0/1 범주로 인코딩 (결과: X, y, df)
# 힌트: load_diabetes(scaled=False, as_frame=True) / (X['sex']==2.0).astype(int)
# ↓ 아래에 직접 작성



## 2. 심화 탐색적 분석(EDA)

### 2.1 타깃 분포와 정규성

왜도·첨도로 분포의 치우침 정량화.

In [ ]:
# [TODO 2] 타깃 분포와 왜도·첨도 확인
# 힌트: skew(y), kurtosis(y) / px.histogram(df, x='target', nbins=30, marginal='box')
# ↓ 아래에 직접 작성



### 2.2 성별·나이 분석 (정규화 해제 후 범주형/실수형 복원)

정규화 상태로는 해석 불가했던 sex(범주형)와 age(나이, 세)를 원본 스케일로 복원해 개별 분석. sex는 1/2 → 그룹 A/B 범주로, age는 연령대로 구간화해 타깃과의 관계 확인.

In [ ]:
# [TODO 3] 성별(범주형) EDA (df_eda 생성, 성별 그룹별 타깃 비교)
# 힌트: X['sex'].map({0:'그룹 A',1:'그룹 B'}) / df_eda.groupby('성별')['target'].agg(...) / px.box(...)
# ↓ 아래에 직접 작성



In [ ]:
# [TODO 4] 나이 EDA 및 연령대 구간화
# 힌트: px.histogram(df, x='age') / px.scatter(..., trendline='ols') / pd.cut(X['age'], bins=[0,40,50,60,120])
# ↓ 아래에 직접 작성



### 2.3 타깃 구간별 피처 분포

타깃을 사분위로 나눠 피처가 구간별로 어떻게 달라지는지 확인.

In [ ]:
# [TODO 5] 타깃 사분위 구간별 핵심 피처 분포(바이올린)
# 힌트: pd.qcut(df['target'], 4, labels=[...]) / px.violin(..., box=True)
# ↓ 아래에 직접 작성



### 2.4 상관 구조와 다중공선성

계층 클러스터맵으로 유사 변수 군집 확인 후, VIF로 공선성 진단.

In [ ]:
# [TODO 6] 상관관계 계층 클러스터맵
# 힌트: df.corr() / sns.clustermap(corr, annot=True, cmap='RdBu_r', center=0)
# ↓ 아래에 직접 작성



In [ ]:
# [TODO 7] 다중공선성 진단(VIF)
# 힌트: add_constant(X) / variance_inflation_factor(Xc.values, i)
# ↓ 아래에 직접 작성



혈청 지표(s1·s2 등)는 서로 강하게 연관되어 VIF가 높게 나타남 — 정규화 모델이 유리한 근거.

### 2.5 비선형 의존성(상호정보량)

선형 상관이 못 잡는 비선형 관계를 MI로 보완.

In [ ]:
# [TODO 8] 상호정보량(MI)으로 비선형 의존성 측정
# 힌트: mutual_info_regression(X, y, random_state=RANDOM_STATE) / px.bar(...)
# ↓ 아래에 직접 작성



### 2.6 차원 축소 및 이상치 탐지

PCA로 구조를 2D로 압축하고 IsolationForest로 이상치 식별.

In [ ]:
# [TODO 9] PCA 2차원 투영(타깃 색칠)
# 힌트: StandardScaler().fit_transform(X) / PCA(n_components=2) / px.scatter(...)
# ↓ 아래에 직접 작성



In [ ]:
# [TODO 10] IsolationForest 이상치 탐지
# 힌트: IsolationForest(contamination=0.05).fit_predict(Xs) / px.scatter(...)
# ↓ 아래에 직접 작성



## 3. 피처 엔지니어링

EDA에서 영향력이 큰 bmi·s5를 중심으로 상호작용·비선형 파생변수 생성.

In [ ]:
# [TODO 11] 파생변수 생성 함수 add_features 정의 후 Xfe 생성
# 힌트: bmi*s5, bmi*bp, s1-s3, bmi**2 등 / Xfe = add_features(X)
# ↓ 아래에 직접 작성



## 4. 모델링: 원본 vs 파생 피처

7종 모델을 RepeatedKFold(5×3)로 평가해 파생변수의 효과 검증.

In [ ]:
# [TODO 12] 다중 모델 벤치마크(원본 vs 파생 피처)
# 힌트: make_models() / RepeatedKFold(5,3) / cross_val_score(m, Xd, y, cv, scoring='r2')
# ↓ 아래에 직접 작성



In [ ]:
# [TODO 13] 원본 vs 파생 성능 비교 시각화
# 힌트: cmp.melt(id_vars='모델', ...) / px.bar(..., barmode='group')
# ↓ 아래에 직접 작성



## 5. 데이터 증강(뻥튀기)과 엄격한 비교

442건은 적은 편이라 학습 데이터 증강을 시도. **핵심 원칙**: 증강은 학습 폴드에만 적용하고 테스트 폴드는 항상 원본 유지 → 데이터 누수 차단. 증강이 실제로 성능을 올리는지 동일 교차검증으로 정직하게 비교.

In [ ]:
# [TODO 14] 데이터 증강 함수 정의(가우시안 노이즈 / GMM 합성)
# 힌트: np.random.RandomState / GaussianMixture(n_components=8).fit(Z).sample(n_new)
# ↓ 아래에 직접 작성



In [ ]:
# [TODO 15] 누수 없는 증강 비교(증강은 학습 폴드만, 테스트는 원본)
# 힌트: KFold(5) / 학습폴드 concat 증강 / r2_score(yte, m.predict(Xte))
# ↓ 아래에 직접 작성



> 해석 주의: 표 형식 회귀에서 합성 증강은 분포를 모방할 뿐 새로운 정보를 만들지 못해, 성능이 크게 오르지 않거나 오히려 소폭 하락하기도 함. 증강은 만능이 아니며 검증으로 확인하는 자세가 중요.

## 6. 하이퍼파라미터 튜닝

최고 성능 계열(HistGBM)에 RandomizedSearchCV 적용.

In [ ]:
# [TODO 16] 하이퍼파라미터 튜닝(HistGBM)
# 힌트: RandomizedSearchCV(HistGradientBoostingRegressor(), param, n_iter=25, cv=5, scoring='r2')
# ↓ 아래에 직접 작성



## 7. 모델 해석

### 7.1 순열 중요도

In [ ]:
# [TODO 17] 최종 모델 학습 및 순열 중요도
# 힌트: train_test_split / best=search.best_estimator_ / permutation_importance(best, Xte, yte, n_repeats=20)
# ↓ 아래에 직접 작성



### 7.2 잔차 분석

In [ ]:
# [TODO 18] 잔차 분석(예측값 대 잔차)
# 힌트: resid = yte - pred / px.scatter(x=pred, y=resid) / fig.add_hline(0)
# ↓ 아래에 직접 작성



### 7.3 부분의존도(PDP)

In [ ]:
# [TODO 19] 부분의존도(PDP)
# 힌트: PartialDependenceDisplay.from_estimator(best, Xtr, top3)
# ↓ 아래에 직접 작성



### 7.4 학습곡선

In [ ]:
# [TODO 20] 학습곡선
# 힌트: learning_curve(best, Xfe, y, cv=5, scoring='r2', train_sizes=np.linspace(0.1,1.0,8))
# ↓ 아래에 직접 작성



## 8. 결론 및 시사점

- 정규화를 풀어 age(나이)·sex(성별)를 해석 가능한 범주/실수로 복원 → 성별·연령대별 타깃 차이를 직접 확인
- bmi와 s5가 선형·비선형·중요도 분석 전반에서 일관되게 핵심 인자로 확인됨
- 혈청 지표 간 강한 공선성 존재 → 정규화 선형 모델 또는 트리 계열이 안정적
- 파생변수(상호작용·비선형 항)는 모델에 따라 소폭의 성능 향상 기여
- 데이터 증강은 누수 없는 비교에서 뚜렷한 개선을 주지 못함 → 표 형식 회귀에서 합성 증강의 한계 확인
- 학습곡선의 검증 성능이 평탄 → 표본 수보다 피처 정보량이 성능의 병목
- 실무 결론: 무리한 증강보다 양질의 피처 확보와 적절한 정규화·튜닝이 우선